# ⚽ FootPredict-Pro — Upcoming Match Predictions
## Week of 29 March – 5 April 2026

This notebook generates **best-model predictions** for all major football fixtures
scheduled in the period **29 March 2026 (today) through 5 April 2026**.

### Coverage
| Date | Competition |
|------|-------------|
| 29 Mar | 🌍 FIFA World Cup 2026 Qualifiers — UEFA & CONMEBOL |
| 1–2 Apr | 🏆 UEFA Champions League Quarter-finals Leg 1 |
| 3 Apr | 🇫🇷 Ligue 1 (Friday fixture) |
| 4 Apr | 🏴󠁧󠁢󠁥󠁮󠁧󠁿 Premier League · 🇪🇸 La Liga · 🇩🇪 Bundesliga · 🇮🇹 Serie A |
| 5 Apr | 🏴󠁧󠁢󠁥󠁮󠁧󠁿 Premier League · 🇮🇹 Serie A · 🇪🇸 La Liga · 🇫🇷 Ligue 1 |

### Model
The **MasterEnsemble** combines:
- **Dixon-Coles bivariate Poisson** (60 % weight) — fitted with team-strength priors derived from 2024-25 season form
- **XGBoost + LightGBM + CatBoost + Logistic Regression stack** (40 % weight, when trained models are available)

All probabilities are calibrated; scoreline distributions are corrected for the
low-score correlation artefact via the Dixon-Coles τ term.

> **Note:** When pre-trained `.joblib` model files are not present the system
> automatically falls back to the seeded Dixon-Coles model (100 % Poisson weight).
> Predictions are still informative — they reflect the built-in team-strength
> priors rather than blank league averages.

In [ ]:
# ── Standard setup ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Ensure FootPredict-Pro is on the Python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.inference.predict_upcoming import (
    predict_all_upcoming,
    print_predictions_report,
    results_to_json,
    STATIC_FIXTURES,
)

print(f"Total static fixtures loaded: {len(STATIC_FIXTURES)}")

## 1 · Run Predictions for the Full Week

In [ ]:
results = predict_all_upcoming(
    date_from="2026-03-29",
    date_to="2026-04-05",
    # model_dir="../models"  # uncomment if trained .joblib files exist
)
print(f"Predictions generated for {len(results)} fixtures.")

In [ ]:
print_predictions_report(results, color=False)

## 2 · Predictions by Competition

### 🌍 WC 2026 Qualifiers — 29 March 2026

In [ ]:
wc_results = predict_all_upcoming(
    date_from="2026-03-29",
    date_to="2026-03-29",
    competition_filter="qualifier",
)
print_predictions_report(wc_results, color=False)

### 🏆 Champions League Quarter-finals (1–2 April 2026)

In [ ]:
ucl_results = predict_all_upcoming(
    date_from="2026-04-01",
    date_to="2026-04-02",
    competition_filter="champions",
)
print_predictions_report(ucl_results, color=False)

### 🏴󠁧󠁢󠁥󠁮󠁧󠁿 Premier League (4–5 April 2026)

In [ ]:
pl_results = predict_all_upcoming(
    date_from="2026-04-04",
    date_to="2026-04-05",
    competition_filter="premier",
)
print_predictions_report(pl_results, color=False)

### 🇮🇹 Serie A (4–5 April 2026) — Derby della Madonnina 🔥

In [ ]:
sa_results = predict_all_upcoming(
    date_from="2026-04-04",
    date_to="2026-04-05",
    competition_filter="serie a",
)
print_predictions_report(sa_results, color=False)

## 3 · Full-week Probability Summary Table

In [ ]:
import pandas as pd

rows = []
for fix, pred in results:
    winner = (
        pred.home_team if pred.p_home_win == max(pred.p_home_win, pred.p_draw, pred.p_away_win)
        else ("Draw" if pred.p_draw == max(pred.p_home_win, pred.p_draw, pred.p_away_win)
              else pred.away_team)
    )
    rows.append({
        "Date":        fix["date"],
        "Competition": fix["competition"],
        "Home":        pred.home_team,
        "Away":        pred.away_team,
        "P(Home)": f"{pred.p_home_win:.1%}",
        "P(Draw)": f"{pred.p_draw:.1%}",
        "P(Away)": f"{pred.p_away_win:.1%}",
        "xG Home":     round(pred.home_xg, 2),
        "xG Away":     round(pred.away_xg, 2),
        "Top Score":   f"{pred.top_scorelines[0][0]}-{pred.top_scorelines[0][1]}" if pred.top_scorelines else "-",
        "Predicted":   winner,
        "Confidence":  pred.confidence,
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 30)
df

## 4 · Visualise Outcome Probabilities

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Filter to the highest-profile fixtures for a cleaner chart ──────────────
highlight_competitions = [
    "UEFA Champions League QF Leg 1",
    "Premier League",
    "Serie A",
]
highlight = [(f, p) for f, p in results if f["competition"] in highlight_competitions]

labels   = [f"{p.home_team}\nvs\n{p.away_team}" for _, p in highlight]
p_home   = [p.p_home_win for _, p in highlight]
p_draw   = [p.p_draw     for _, p in highlight]
p_away   = [p.p_away_win for _, p in highlight]

x   = np.arange(len(labels))
w   = 0.28

fig, ax = plt.subplots(figsize=(max(14, len(labels) * 1.5), 6))

bars_h = ax.bar(x - w,  p_home, w, label="Home Win", color="#2196F3", alpha=0.85)
bars_d = ax.bar(x,      p_draw, w, label="Draw",     color="#9E9E9E", alpha=0.85)
bars_a = ax.bar(x + w,  p_away, w, label="Away Win", color="#F44336", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Probability", fontsize=11)
ax.set_title(
    "FootPredict-Pro — Match Outcome Probabilities\n"
    "Champions League QF · Premier League · Serie A  (29 Mar – 5 Apr 2026)",
    fontsize=12, fontweight="bold",
)
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("../data/predictions_chart_2026-03-29.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/predictions_chart_2026-03-29.png")

## 5 · UCL Quarter-final Deep Dive

In [ ]:
from src.inference.predict_upcoming import _build_seeded_model
from src.models.ensemble import MasterEnsemble

dc   = _build_seeded_model()
ens  = MasterEnsemble(dixon_coles_model=dc, outcome_model=None, player_model=None)

ucl_fixtures = [
    ("Real Madrid",     "Arsenal"),
    ("Bayern Munich",   "PSG"),
    ("Manchester City", "Inter Milan"),
    ("Barcelona",       "Atletico Madrid"),
]

print("\n🏆 UEFA Champions League Quarter-finals — Scoreline Distributions\n")
for home, away in ucl_fixtures:
    pred = ens.predict(home_team=home, away_team=away)
    print(f"  {home:20s} vs {away:20s}")
    print(f"    Outcome:  Home {pred.p_home_win:.1%}  |  Draw {pred.p_draw:.1%}  |  Away {pred.p_away_win:.1%}")
    print(f"    xG:       {pred.home_xg:.2f}  –  {pred.away_xg:.2f}")
    print("    Top 5 scorelines:")
    for h, a, p in pred.top_scorelines[:5]:
        print(f"      {h}-{a}  ({p:.1%})")
    print()

## 6 · Export Full Predictions to JSON

In [ ]:
import json
from pathlib import Path

output_path = Path("../data/predictions_2026-03-29_to_2026-04-05.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = results_to_json(results)
output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(f"Exported {len(payload)} predictions to {output_path}")

# Preview first entry
print("\nSample entry:")
print(json.dumps(payload[0], indent=2))

## 7 · How to Run Live (with Real Fixture Data)

Once you have an **API-Football** key set in `config.yaml`, real upcoming fixtures
are fetched automatically and predictions use the fully trained ensemble:

```bash
# Train the models first (requires historical data download)
make fetch-csv LEAGUE=E0      # Premier League CSV
make fetch-csv LEAGUE=SP1     # La Liga CSV
make train LEAGUE=E0 SEASONS="2022 2023 2024"

# Then predict today's + next 7 days' fixtures
make predict-upcoming

# Or with a custom range / JSON output
python src/inference/predict_upcoming.py \\
    --date-from 2026-03-29 --date-to 2026-04-05 \\
    --output-format json --output-file predictions_week.json
```

The script falls back gracefully to the seeded Dixon-Coles when no trained
models or API key are present — so you always get useful predictions.
